# osu! Chat Bot RAG in Google Colab

This notebook is a guided version of the command-line workflow for the osu! chat bot prototype. It is designed for Google Colab, but most cells also work in a local Jupyter notebook.

The goal is to make the heavier RAG pipeline easy to inspect and rerun without asking your laptop to carry the model work or a huge stream of notebook output.

The project currently behaves like a RAG system rather than a model fine-tuning project. This notebook now treats dense indexing and generative entity extraction as the main path:

1. Parse the osu! wiki/news corpus into structured documents and retrieval chunks.
2. Build link artifacts used for review and diagnostics.
3. Build a dense Qdrant index with sentence-transformer embeddings.
4. Extract and normalize generative entity candidates with GLiNER.
5. Evaluate retrieval quality with dense retrieval enabled.
6. Use keyword-only retrieval only as a quick diagnostic baseline.

Long-running stages start as background jobs, write logs under `artifacts/rag/jobs`, and show only small log tails in the notebook.


## 0. Colab Runtime Recommendation

Use a GPU runtime for the main workflow. Dense embeddings and GLiNER both run on CPU, but a CPU-only runtime is mostly useful for quick artifact checks and keyword diagnostics.

Recommended Colab setting:

- `Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU` or better.

The notebook automatically sets the embedding device to `cuda` when PyTorch can see a GPU. GLiNER uses its own model loading path, so keep an eye on the job log during the first model download.


In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Running in Colab: {IN_COLAB}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
except Exception as exc:
    print(f"PyTorch check skipped: {exc}")

## 1. Get the Project Code

Colab opens a notebook by itself; it does not automatically clone the whole repository that contains the notebook. Set `REPO_URL` to your GitHub repository URL before running this cell.

Example:

```python
REPO_URL = "https://github.com/your-name/osu-chat-bot.git"
```

If you are running this notebook locally from the project root, leave `REPO_URL` empty.

In [ ]:
# TODO: In Colab, set this to your repository URL.
REPO_URL = ""

PROJECT_DIR = Path("/content/osu-chat-bot") if IN_COLAB else Path.cwd()

if IN_COLAB and not (PROJECT_DIR / "pyproject.toml").exists():
    if not REPO_URL:
        raise ValueError(
            "Set REPO_URL to your GitHub repo URL, then rerun this cell. "
            "Example: REPO_URL = 'https://github.com/your-name/osu-chat-bot.git'"
        )
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print(f"Working directory: {Path.cwd()}")
print(f"pyproject.toml exists: {(Path.cwd() / 'pyproject.toml').exists()}")

## 2. Install Python Dependencies

The base install includes the retrieval and indexing dependencies used by the main workflow:

- `sentence-transformers` for dense embeddings.
- `qdrant-client` for the local vector store.
- `pytest` from the `dev` extra, useful if you want to run tests.

The GLiNER extra is installed inside the background entity job. That keeps the setup cell quiet and puts any large dependency output into the job log instead of the notebook.


In [ ]:
%pip install -q -e ".[dev]"

## 3. Download the osu! Wiki Corpus

The chatbot reads from a local checkout of `ppy/osu-wiki`. The default config expects it at `database/osu-wiki`.

The clone is shallow to keep Colab setup quick. If you need full git history later, remove `--depth 1`.

In [ ]:
OSU_WIKI_DIR = Path("database/osu-wiki")

if not OSU_WIKI_DIR.exists():
    OSU_WIKI_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ppy/osu-wiki.git", str(OSU_WIKI_DIR)],
        check=True,
    )
else:
    print(f"Corpus already exists: {OSU_WIKI_DIR}")

print(f"Corpus path: {OSU_WIKI_DIR.resolve()}")

## 4. Write a Colab-Friendly Config

This config keeps artifacts inside the project folder so they are easy to zip and download later.

The embedding device is selected automatically:

- `cuda` when Colab has a GPU.
- `cpu` otherwise.

The local Qdrant URL stores vectors under `artifacts/rag/qdrant`. This is simple and portable for notebook experiments.

In [ ]:
try:
    import torch
    EMBEDDING_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    EMBEDDING_DEVICE = "cpu"

HF_CACHE_DIR = "/content/hf-cache" if IN_COLAB else "artifacts/hf-cache"

CONFIG_PATH = Path("config.colab.toml")
CONFIG_PATH.write_text(
    f"""
[corpus]
osu_wiki_path = "database/osu-wiki"
language = "en"
include_news = true

[artifacts]
path = "artifacts/rag"

[embedding]
model = "sentence-transformers/all-MiniLM-L6-v2"
cache_folder = "{HF_CACHE_DIR}"
device = "{EMBEDDING_DEVICE}"
local_files_only = false

[qdrant]
url = "file://artifacts/rag/qdrant"
collection = "osu_wiki_en"
vector_size = 384

[retrieval]
dense_top_k = 24
final_top_k = 6

[ollama]
url = "http://127.0.0.1:11434"
model = "mistral"
temperature = 0.1
""".strip()
    + "\n",
    encoding="utf-8",
)

print(CONFIG_PATH.read_text())

## 5. Helper Functions for Running `osu-bot`

The project exposes a CLI named `osu-bot`. In notebooks, it is useful to wrap commands so every command uses the same config file and report paths.

Example command from a terminal:

```bash
osu-bot --config config.colab.toml stats
```

Equivalent foreground notebook call:

```python
run_osu_bot("stats")
```

For long jobs, use `start_osu_bot_job(...)` in the next section so output goes to a log file.


In [ ]:
import json
import shlex
import subprocess

ARTIFACT_DIR = Path("artifacts/rag")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

EVAL_DATASET = Path("eval/osu_seed.jsonl")
KEYWORD_EVAL_REPORT = ARTIFACT_DIR / "eval_seed_keyword_report.json"
DENSE_EVAL_REPORT = ARTIFACT_DIR / "eval_seed_dense_report.json"

def run_osu_bot(*args, check=True):
    cmd = ["osu-bot", "--config", str(CONFIG_PATH), *map(str, args)]
    print("$ " + " ".join(shlex.quote(part) for part in cmd))
    return subprocess.run(cmd, check=check)

def show_json(path, max_chars=4000):
    path = Path(path)
    data = json.loads(path.read_text(encoding="utf-8"))
    text = json.dumps(data, indent=2, ensure_ascii=False)
    print(text[:max_chars])
    if len(text) > max_chars:
        print(f"\n... truncated after {max_chars} characters ...")
    return data


## 6. Build Required Source Artifacts

These commands prepare the files that both dense retrieval and generative NER depend on.

### `ingest`

Parses English wiki pages and news posts into structured documents and retrieval chunks.

Outputs include:

- `documents_structured.jsonl`
- `chunks_hierarchical.jsonl`
- `ingest_report.json`

### `links`

Builds reviewable hyperlink alias artifacts. This helps inspect which link texts are useful aliases and which are noisy.

The cell below runs the full preparation chain in the background and keeps notebook output limited to a compact log tail.


## Background Jobs for Long Runs

Colab notebooks can become uncomfortable when a long command is tied directly to a cell. The helpers below run CLI work as background processes and write output to log files.

Use this for preparation, dense indexing, dense evaluation, and GLiNER entity extraction. The notebook stays responsive, and `wait_for_job(...)` refreshes a small log tail instead of accumulating thousands of output lines.

The helper writes:

- `artifacts/rag/jobs/<job>.pid`
- `artifacts/rag/jobs/<job>.status`
- `artifacts/rag/jobs/<job>.log`
- `artifacts/rag/jobs/<job>.cmd.txt`


In [ ]:
import subprocess
import shlex
import time
from pathlib import Path

try:
    from IPython.display import clear_output
except Exception:
    clear_output = None

JOB_DIR = ARTIFACT_DIR / "jobs"
JOB_DIR.mkdir(parents=True, exist_ok=True)

def osu_bot_shell(*args):
    cmd = ["osu-bot", "--config", str(CONFIG_PATH), *map(str, args)]
    return " ".join(shlex.quote(part) for part in cmd)

def _pid_running(pid):
    if not pid:
        return False
    check = subprocess.run(
        ["bash", "-lc", f"ps -p {shlex.quote(str(pid))} >/dev/null"],
        text=True,
    )
    return check.returncode == 0

def job_exit_code(job_name):
    status_path = JOB_DIR / f"{job_name}.status"
    if not status_path.exists():
        return None
    raw = status_path.read_text(encoding="utf-8").strip()
    return int(raw) if raw else None

def start_background_job(job_name, shell_cmd):
    job_name = str(job_name)
    pid_path = JOB_DIR / f"{job_name}.pid"
    status_path = JOB_DIR / f"{job_name}.status"
    log_path = JOB_DIR / f"{job_name}.log"
    cmd_path = JOB_DIR / f"{job_name}.cmd.txt"

    if pid_path.exists():
        old_pid = pid_path.read_text(encoding="utf-8").strip()
        if _pid_running(old_pid):
            raise RuntimeError(f"Job {job_name!r} is already running with PID {old_pid}.")

    if status_path.exists():
        status_path.unlink()

    cmd_path.write_text(shell_cmd + "\n", encoding="utf-8")
    wrapped_cmd = (
        "set -o pipefail; "
        f"{shell_cmd}; "
        "code=$?; "
        f"echo $code > {shlex.quote(str(status_path))}; "
        "exit $code"
    )
    launcher = (
        f"nohup bash -lc {shlex.quote(wrapped_cmd)} "
        f"> {shlex.quote(str(log_path))} 2>&1 & "
        f"echo $! > {shlex.quote(str(pid_path))}"
    )
    subprocess.run(["bash", "-lc", launcher], check=True)

    pid = pid_path.read_text(encoding="utf-8").strip()
    print(f"Started {job_name}: PID {pid}")
    print(f"Command: {cmd_path}")
    print(f"Log: {log_path}")
    return {"job": job_name, "pid": pid, "log": log_path, "command": cmd_path}

def start_osu_bot_job(job_name, *args, pre_command=None, post_command=None):
    shell_cmd = osu_bot_shell(*args)
    if pre_command:
        shell_cmd = f"{pre_command} && {shell_cmd}"
    if post_command:
        shell_cmd = f"{shell_cmd} && {post_command}"
    return start_background_job(job_name, shell_cmd)

def job_status(job_name):
    pid_path = JOB_DIR / f"{job_name}.pid"
    if not pid_path.exists():
        print(f"No PID file for {job_name!r}.")
        return None
    pid = pid_path.read_text(encoding="utf-8").strip()
    code = job_exit_code(job_name)
    if code is not None:
        print(f"{job_name}: finished with exit code {code} (PID {pid})")
    else:
        subprocess.run(["bash", "-lc", f"ps -p {shlex.quote(pid)} -o pid,etime,cmd"])
    return pid

def show_job_log(job_name, chars=4000):
    log_path = JOB_DIR / f"{job_name}.log"
    if not log_path.exists():
        print(f"No log file for {job_name!r}: {log_path}")
        return ""
    text = log_path.read_text(encoding="utf-8", errors="replace")
    tail = text[-chars:]
    print(tail)
    return tail

def wait_for_job(job_name, poll_seconds=30, tail_chars=3000, raise_on_error=True):
    pid_path = JOB_DIR / f"{job_name}.pid"
    if not pid_path.exists():
        raise RuntimeError(f"No PID file for {job_name!r}.")
    pid = pid_path.read_text(encoding="utf-8").strip()

    while job_exit_code(job_name) is None and _pid_running(pid):
        if clear_output:
            clear_output(wait=True)
        print(f"Waiting for {job_name} (PID {pid}). Refreshing every {poll_seconds}s.")
        show_job_log(job_name, chars=tail_chars)
        time.sleep(poll_seconds)

    if clear_output:
        clear_output(wait=True)
    job_status(job_name)
    show_job_log(job_name, chars=tail_chars)
    code = job_exit_code(job_name)
    if raise_on_error and code not in (None, 0):
        raise RuntimeError(f"Job {job_name!r} failed with exit code {code}. See {JOB_DIR / (job_name + '.log')}.")
    return code

def stop_job(job_name):
    pid_path = JOB_DIR / f"{job_name}.pid"
    if not pid_path.exists():
        print(f"No PID file for {job_name!r}.")
        return
    pid = pid_path.read_text(encoding="utf-8").strip()
    subprocess.run(["bash", "-lc", f"kill {shlex.quote(pid)}"])
    print(f"Stop requested for {job_name}: PID {pid}")


In [ ]:
RUN_PREPARE_ARTIFACTS = True
WAIT_FOR_PREPARE_ARTIFACTS = True

if RUN_PREPARE_ARTIFACTS:
    prepare_cmd = " && ".join([
        osu_bot_shell("ingest"),
        osu_bot_shell("links"),
    ])
    start_background_job("prepare_artifacts", prepare_cmd)
    if WAIT_FOR_PREPARE_ARTIFACTS:
        wait_for_job("prepare_artifacts", poll_seconds=20, tail_chars=2500)
    else:
        print("Check progress with job_status('prepare_artifacts') and show_job_log('prepare_artifacts').")
else:
    print("Preparation skipped. Set RUN_PREPARE_ARTIFACTS = True to rebuild source artifacts.")


## 7. Inspect Statistics and Validation

Run these before indexing or evaluating. They give you a quick health check.

- `stats` summarizes document, chunk, structure, and term counts.
- `validate` reports possible artifact problems.

Validation severities mean:

- `error`: fix before trusting downstream results.
- `warning`: inspect because retrieval quality may be affected.
- `info`: useful corpus notes, often acceptable.

This health check also runs in the background so validation output lands in a log file.


In [ ]:
RUN_HEALTH_CHECK = True
WAIT_FOR_HEALTH_CHECK = True

if RUN_HEALTH_CHECK:
    health_cmd = f"{osu_bot_shell('stats')} ; {osu_bot_shell('validate')}"
    start_background_job("health_check", health_cmd)
    if WAIT_FOR_HEALTH_CHECK:
        code = wait_for_job("health_check", poll_seconds=10, tail_chars=2500, raise_on_error=False)
        if code not in (None, 0):
            print("Validation reported errors. Inspect validation_report.json before trusting downstream results.")
    else:
        print("Check progress with job_status('health_check') and show_job_log('health_check').")
else:
    print("Health check skipped.")


In [ ]:
if (ARTIFACT_DIR / "stats_report.json").exists():
    stats_report = show_json(ARTIFACT_DIR / "stats_report.json", max_chars=2500)
else:
    print("stats_report.json is not available yet.")

if (ARTIFACT_DIR / "validation_report.json").exists():
    validation_report = show_json(ARTIFACT_DIR / "validation_report.json", max_chars=2500)
else:
    print("validation_report.json is not available yet.")


## 8. Keyword-Only Smoke Check

`inspect --keyword-only` does not require Qdrant or embeddings. It is no longer the main retrieval path for this notebook, but it is still a useful smoke test when source artifacts look suspicious.

Keep this off during full runs to avoid filling the notebook with repeated retrieval output.


In [ ]:
RUN_KEYWORD_SMOKE = False

example_questions = [
    "What is a beatmap?",
    "What does AR change in osu?",
    "How does a beatmap become ranked?",
    "osu feels choppy and my frames are awful, what should I check?",
]

if RUN_KEYWORD_SMOKE:
    for question in example_questions:
        print("\n" + "=" * 100)
        print(question)
        print("=" * 100)
        run_osu_bot("inspect", question, "--keyword-only")
else:
    print("Keyword smoke skipped. Set RUN_KEYWORD_SMOKE = True for a quick diagnostic check.")


## 9. Optional: Keyword Baseline Evaluation

The seed dataset lives at `eval/osu_seed.jsonl`. Each row has:

- `category`: a rough question group.
- `question`: the user-style prompt.
- `expected_document_ids`: one or more wiki pages that should appear in retrieval.

Example row:

```json
{"category": "definition", "question": "wait, what even is a beatmap in osu?", "expected_document_ids": ["Beatmap"]}
```

Dense evaluation is the main score to watch. Keyword-only evaluation is kept as a baseline comparison and runs in the background when enabled.


In [ ]:
RUN_KEYWORD_EVAL = False
WAIT_FOR_KEYWORD_EVAL = True

if RUN_KEYWORD_EVAL:
    start_osu_bot_job("keyword_eval", "eval", EVAL_DATASET, "--output", KEYWORD_EVAL_REPORT)
    if WAIT_FOR_KEYWORD_EVAL:
        wait_for_job("keyword_eval", poll_seconds=10, tail_chars=2500)
    else:
        print("Check progress with job_status('keyword_eval') and show_job_log('keyword_eval').")
else:
    print("Keyword evaluation skipped. Set RUN_KEYWORD_EVAL = True if you want the baseline report.")


In [ ]:
import pandas as pd

if KEYWORD_EVAL_REPORT.exists():
    keyword_eval = show_json(KEYWORD_EVAL_REPORT, max_chars=3000)
    rows = keyword_eval["examples"]
    df = pd.DataFrame(rows)
    display(df[["category", "question", "matched", "top_document_ids"]].head(12))
else:
    print(f"No keyword report yet: {KEYWORD_EVAL_REPORT}")


## 10. Mainline: Build a Dense Index

Dense indexing embeds chunks with `sentence-transformers/all-MiniLM-L6-v2` and writes vectors to local Qdrant storage.

This is now the primary retrieval setup step. The default below indexes the full artifact set with `--resume` so interrupted Colab sessions can continue from `artifacts/rag/index_state.json`.

For a quick trial, set `INDEX_LIMIT = 500`. For the main run, leave `INDEX_LIMIT = None`.

The job runs in the background and logs progress to `artifacts/rag/jobs/index.log`.


In [ ]:
RUN_DENSE_INDEXING = True
WAIT_FOR_DENSE_INDEXING = True
INDEX_LIMIT = None
INDEX_BATCH_SIZE = 64

if RUN_DENSE_INDEXING:
    index_args = ["index", "--resume", "--batch-size", INDEX_BATCH_SIZE]
    if INDEX_LIMIT is not None:
        index_args.extend(["--limit", INDEX_LIMIT])
    start_osu_bot_job("index", *index_args)
    if WAIT_FOR_DENSE_INDEXING:
        wait_for_job("index", poll_seconds=60, tail_chars=3000)
    else:
        print("Check progress with job_status('index') and show_job_log('index').")
else:
    print("Dense indexing skipped. Set RUN_DENSE_INDEXING = True to start it in the background.")


## 11. Mainline: Generative NER and Normalization

GLiNER is now treated as the main enrichment path for discovering entity candidates from chunk text. The old deterministic `terms` artifact is not generated in this notebook and is ignored by dense retrieval.

The default below scans the full chunk artifact, then immediately runs `normalize-entities` in the same background job. That produces reviewable generative entity artifacts without streaming model output into the notebook.

Artifacts:

- `entity_candidates_generative.jsonl`
- `entity_candidates_generative_report.json`
- `entity_normalization.jsonl`
- `entity_normalization_review.csv`
- `entity_normalization_report.json`

For a quick trial, set `ENTITY_LIMIT = 100`. For the main run, leave `ENTITY_LIMIT = None`.


In [ ]:
RUN_GENERATIVE_NER = True
WAIT_FOR_GENERATIVE_NER = True
ENTITY_LIMIT = None
ENTITY_THRESHOLD = 0.5
ENTITY_MAX_TEXT_CHARS = 3500

if RUN_GENERATIVE_NER:
    entity_args = [
        "entities",
        "--backend", "gliner",
        "--label-profile", "main-page",
        "--sampling", "balanced",
        "--threshold", ENTITY_THRESHOLD,
        "--max-text-chars", ENTITY_MAX_TEXT_CHARS,
    ]
    if ENTITY_LIMIT is not None:
        entity_args.extend(["--limit", ENTITY_LIMIT])

    pre_command = f"{shlex.quote(sys.executable)} -m pip install -q -e '.[dev,entities]'"
    post_command = osu_bot_shell("normalize-entities")
    start_osu_bot_job(
        "generative_entities",
        *entity_args,
        pre_command=pre_command,
        post_command=post_command,
    )
    if WAIT_FOR_GENERATIVE_NER:
        wait_for_job("generative_entities", poll_seconds=60, tail_chars=3000)
    else:
        print("Check progress with job_status('generative_entities') and show_job_log('generative_entities').")
else:
    print("Generative NER skipped. Set RUN_GENERATIVE_NER = True to start it in the background.")


## 12. Mainline: Dense Retrieval Evaluation

Run this after the dense index is built. Dense retrieval combines Qdrant results with the keyword/entity scoring pipeline.

This is the notebook equivalent of the cluster `eval_dense` task and is the main retrieval report to compare between runs.


In [ ]:
RUN_DENSE_EVAL = True
WAIT_FOR_DENSE_EVAL = True

if RUN_DENSE_EVAL:
    start_osu_bot_job("dense_eval", "eval", EVAL_DATASET, "--dense", "--output", DENSE_EVAL_REPORT)
    if WAIT_FOR_DENSE_EVAL:
        wait_for_job("dense_eval", poll_seconds=15, tail_chars=3000)
        if DENSE_EVAL_REPORT.exists():
            dense_eval = show_json(DENSE_EVAL_REPORT, max_chars=3000)
    else:
        print("Check progress with job_status('dense_eval') and show_job_log('dense_eval').")
        print("After it finishes, run: dense_eval = show_json(DENSE_EVAL_REPORT, max_chars=3000)")
else:
    print("Dense evaluation skipped. Set RUN_DENSE_EVAL = True after indexing.")


## 13. About Answer Generation in Colab

The CLI command `query` expects a local Ollama server at `http://127.0.0.1:11434` and defaults to the `mistral` model.

That is convenient on a laptop, but less convenient in Colab because Colab sessions are temporary and do not ship with Ollama running.

For Colab, the most reliable workflow is:

1. Use dense `inspect` after indexing to verify retrieved sources.
2. Use dense `eval` to measure retrieval quality.
3. Download artifacts.
4. Run `query` locally later, or adapt the generation layer to call a hosted model API.

Keyword-only inspection is still available as a diagnostic fallback, but dense inspection is the path to check before answer generation.

In [ ]:
RUN_DENSE_INSPECT = True
question = "What does osu!supporter unlock?"

if RUN_DENSE_INSPECT:
    run_osu_bot("inspect", question)
else:
    print("Dense inspect skipped. Set RUN_DENSE_INSPECT = True after indexing.")


## 14. Save or Download Artifacts

Colab files disappear when the runtime is recycled. Download your artifacts before closing the session.

The zip file includes reports, JSONL artifacts, CSV review files, job logs, and the local Qdrant folder from dense indexing.

In [ ]:
import zipfile

ZIP_PATH = Path("artifacts_rag_colab.zip")

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in ARTIFACT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, path.as_posix())

print(f"Wrote {ZIP_PATH.resolve()} ({ZIP_PATH.stat().st_size / 1_000_000:.2f} MB)")

if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP_PATH))

## 15. Optional: Persist Artifacts to Google Drive

Downloading a zip is simple, but Drive is nicer for longer experiments. This cell mounts Drive and copies the artifact zip there.

You can also copy the whole `artifacts/rag` directory if you prefer to keep files uncompressed.

In [ ]:
COPY_TO_DRIVE = False

if COPY_TO_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Google Drive mounting is only available in Colab.")
    from google.colab import drive
    import shutil

    drive.mount("/content/drive")
    drive_dir = Path("/content/drive/MyDrive/osu-chat-bot-artifacts")
    drive_dir.mkdir(parents=True, exist_ok=True)
    destination = drive_dir / ZIP_PATH.name
    shutil.copy2(ZIP_PATH, destination)
    print(f"Copied to {destination}")
else:
    print("Drive copy skipped. Set COPY_TO_DRIVE = True to enable it.")

## 16. Troubleshooting

### `Set REPO_URL...`

Colab needs to clone the project before it can install it. Set `REPO_URL` near the top of the notebook.

### `No chunks found. Run osu-bot ingest first.`

Run the source artifact section again. `inspect`, `query`, `eval`, `index`, and `entities` need `chunks_hierarchical.jsonl`.

### Dense retrieval fails

Make sure the `index` job finished with exit code 0. Keyword-only retrieval does not need Qdrant, but dense retrieval does.

### GLiNER is slow or fails during the first run

Check `show_job_log('generative_entities')`. The first run installs the entity extra and downloads the GLiNER model, so it can spend a while in setup before scanning chunks.

### Hugging Face download is slow or throttled

Use a smaller `INDEX_LIMIT` or `ENTITY_LIMIT` while testing. For serious repeated runs, mount Drive and set `cache_folder` to a persistent Drive path.

### Colab disconnects

Use smaller indexing intervals, for example `INDEX_LIMIT = 500`, then resume with `--resume`. Download or copy artifacts to Drive between long stages.